In [1]:
SEED=15179996

In [2]:
!pip install -q -U transformers peft datasets optuna evaluate scikit-learn accelerate tqdm pandas huggingface_hub torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 137.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 144.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 147.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour 

In [3]:
import torch
import pandas as pd
import numpy as np
import optuna
import evaluate
from datasets import load_dataset, DatasetDict, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DefaultDataCollator
)
from peft import LoraConfig, get_peft_model
from huggingface_hub import HfApi
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Dataset

Sentiment140, 30k train, 1k test

https://huggingface.co/datasets/bdanko/sentiment140

In [4]:
# We increase training set to 30K
# To preserve the same testing set we originally conducted for the optuna baselines
# we will sample first original 5k samples based on our original seed, then the
# 1000 testing, then we get the remaining text we want to train on

def prepare_dataset(dataset_name="bdanko/sentiment140", train_size=30000, test_size=1000):
    print(f"Loading dataset {dataset_name}...")
    dataset = load_dataset(dataset_name, split="train")
    df = dataset.to_pandas()

    df['sentiment'] = df['sentiment'].replace(4, 1)

    original_5k_train = df.sample(n=5000, weights=df['sentiment'].map({0: 0.5, 1: 0.5}), random_state=SEED)
    remaining_pool = df.drop(original_5k_train.index)

    test_df_neg = remaining_pool[remaining_pool['sentiment'] == 0].sample(n=test_size // 2, random_state=SEED)
    test_df_pos = remaining_pool[remaining_pool['sentiment'] == 1].sample(n=test_size // 2, random_state=SEED)
    test_df = pd.concat([test_df_neg, test_df_pos]).sample(frac=1, random_state=SEED)

    train_pool = df.drop(test_df.index)
    train_df = train_pool.sample(n=train_size, random_state=SEED)

    print(f"Test set preserved: {len(test_df)} rows.")
    print(f"New training set: {len(train_df)} rows.")

    return DatasetDict({
        "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
        "test": Dataset.from_pandas(test_df.reset_index(drop=True))
    })

In [5]:
dataset = prepare_dataset()
print(dataset)

Loading dataset bdanko/sentiment140...


README.md:   0%|          | 0.00/420 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/113M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1600000 [00:00<?, ? examples/s]

Test set preserved: 1000 rows.
New training set: 30000 rows.
DatasetDict({
    train: Dataset({
        features: ['text', 'date', 'user', 'sentiment', 'query'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['text', 'date', 'user', 'sentiment', 'query'],
        num_rows: 1000
    })
})


In [6]:
model_id = "mistralai/Mistral-7B-v0.3"
# We stick with the base model,
# because we do not need to care about instruction alignment
# we are training a classifier head that we don't need to explicitly
# care about generation parsing

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [7]:
def tokenize_function(examples):
    texts = [f"Sentiment classification. Text: {text}\nSentiment: " for text in examples["text"]]
    tokenized = tokenizer(texts, truncation=True, max_length=128, padding="max_length")
    tokenized["labels"] = examples["sentiment"]
    return tokenized

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [8]:
def get_base_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=2,
        dtype=torch.bfloat16,
        device_map={"": 0}, # everything must be on one
                            # GPU and namespace or optuna complains
      )
    model.config.pad_token_id = model.config.eos_token_id
    return model

In [9]:
# huggingface standardized metrics for:
# accuracy
# precision
# recll
# f1

accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision_metric.compute(predictions=predictions, references=labels)["precision"],
        "recall": recall_metric.compute(predictions=predictions, references=labels)["recall"],
        "f1": f1_metric.compute(predictions=predictions, references=labels)["f1"],
    }

In [10]:
r=32
lr=0.0005
alpha=32
lora_dropout=0.1

peft_config = LoraConfig(
    r=16, lora_alpha=alpha,
    target_modules=["q_proj", "v_proj"], lora_dropout=lora_dropout,
    bias="none", task_type="SEQ_CLS",
)
lr      = lr
out_dir = "./best_lora"
hub_id  = "bdanko/mistral-7b-sentiment-lora-v2"

torch.cuda.reset_peak_memory_stats()
model = get_base_model()

model.add_adapter(peft_config, adapter_name="best_adapter")

args = TrainingArguments(
    output_dir=out_dir,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    learning_rate=lr,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False, # causes saving errors
    metric_for_best_model="f1",
    bf16=True,
    report_to="none",
    push_to_hub=True,
    hub_model_id=hub_id,
    #gradient_checkpointing=True,
    #gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-v0.3
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.952376,0.696184,0.500000,0.500000,1.000000,0.666667
2,0.581736,0.535834,0.735000,0.788698,0.642000,0.707828
3,0.503507,0.500721,0.758000,0.747126,0.780000,0.763209
4,0.471661,0.482417,0.783000,0.791753,0.768000,0.779695
5,0.466940,0.478683,0.777000,0.791579,0.752000,0.771282


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9375, training_loss=0.5733551708984375, metrics={'train_runtime': 3590.1401, 'train_samples_per_second': 41.781, 'train_steps_per_second': 2.611, 'total_flos': 8.048346660864e+17, 'train_loss': 0.5733551708984375, 'epoch': 5.0})